# E6 | Model Clustering K-Means


In [1]:
# ===== [1] CARGA DE ml.ml_cluster_dataset (banco fiap) =====
import os
import sys
import pandas as pd
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Sobe a arvore ate achar a raiz do repo (marcada por .git ou etl/)."""
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'etl').is_dir():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Saida: artefatos do modelo em data/ml (mantido para os parquets de diagnostico/EDA).
OUT_DIR = Path(os.getenv('ML_OUTPUT_DIR', PROJECT_ROOT / 'data' / 'ml' / 'kmeans'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

from etl.db import get_engine

engine = get_engine()
df_parquet = pd.read_sql('SELECT * FROM ml.ml_cluster_dataset', engine)

# Prefixo de todo artefato de saida: <modelo>_<nome_da_saida>.parquet
MODELO_ARTEFATO = 'kmeans'
MODELO_VERSAO_CLUSTER = 'kmeans_v1'

def salvar_parquet(df: pd.DataFrame, nome: str) -> Path:
    """Grava um artefato em OUT_DIR (data/ml/kmeans) como parquet.

    Colunas 'category'/'object' viram string: o parquet exige tipo homogeneo por coluna.
    """
    saida = df.copy()
    for c in saida.columns:
        if str(saida[c].dtype) == 'category' or saida[c].dtype == object:
            saida[c] = saida[c].astype('string')   # 'string' preserva nulos; str() nao
    caminho_out = OUT_DIR / f'{nome}.parquet'
    saida.to_parquet(caminho_out, index=False)
    print(f'   -> {caminho_out}  ({len(saida):,} linhas)')
    return caminho_out

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Entrada      : ml.ml_cluster_dataset (banco fiap)')
print(f'OUT_DIR      : {OUT_DIR}')
print(f'   {df_parquet.shape[0]:,} linhas x {df_parquet.shape[1]} colunas')


PROJECT_ROOT : /home/fiap/mvp-locaweb
Entrada      : ml.ml_cluster_dataset (banco fiap)
OUT_DIR      : /home/fiap/mvp-locaweb/data/ml/kmeans
   41,441 linhas x 26 colunas


In [2]:
!pip install mlflow

/bin/bash: line 1: pip: command not found


In [3]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np, tempfile, joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


In [4]:
# ===== [3b] EDA: ANALISE DOS DADOS ANTES DE TRANSFORMACAO =====
from pathlib import Path

df = df_parquet.copy() # Adicionei esta linha para definir 'df'

# CAUSA: Features de ENTRADA
cause_cols = [
    'prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto',
    'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial',
    'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta'
]

# EFEITO: Features de SAIDA (EXCLUIR do treino)
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# EXCLUIR: NULL placeholders, redundantes
exclude_cols = [
    'incident_id', 'cluster', 'data_abertura',
    'duracao_horas_scaled', 'cluster_id',
    'is_filho_de_problema'
]

print('[STATS] ESTRUTURA DE FEATURES:')
print('   [OK] CAUSA (incluir): %d colunas' % len(cause_cols))
print('   [WARN] EFEITO (interpretar): %d colunas' % len(effect_cols))
print('   [ERROR] EXCLUIR: %d colunas' % len(exclude_cols))
print('   Total dataset: %d' % len(df.columns))

# ===== EDA em parquet: os numeros que antes viravam grafico =====
# Cada bloco abaixo corresponde a um painel da figura antiga.

cat_cols_all = df.select_dtypes(include=['object']).columns.tolist()
eda_cardinalidade = (pd.DataFrame({'coluna': cat_cols_all,
                                   'n_valores_unicos': [df[c].nunique() for c in cat_cols_all]})
                     .sort_values('n_valores_unicos', ascending=False))
salvar_parquet(eda_cardinalidade, 'eda_cardinalidade')

cause_numeric = [c for c in cause_cols if c in df.columns and df[c].dtype in ['int64', 'float64']]
if cause_numeric:
    eda_estatisticas = df[cause_numeric].describe().T.reset_index().rename(columns={'index': 'feature'})
    salvar_parquet(eda_estatisticas, 'eda_estatisticas_causa')

if len(cause_numeric) > 1:
    # matriz de correlacao em formato longo: um par de features por linha
    eda_correlacao = (df[cause_numeric].corr()
                      .stack().rename('correlacao').reset_index()
                      .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'}))
    salvar_parquet(eda_correlacao, 'eda_correlacao_causa')

nulos = df.isnull().sum()
eda_nulos = (pd.DataFrame({'coluna': nulos.index, 'n_nulos': nulos.values})
             .assign(pct_nulos=lambda d: 100 * d.n_nulos / len(df))
             .query('n_nulos > 0').sort_values('pct_nulos', ascending=False))
salvar_parquet(eda_nulos, 'eda_nulos')

if 'categoria' in df.columns:
    vc = df['categoria'].value_counts()
    eda_categorias = pd.DataFrame({'categoria': vc.index, 'n_incidentes': vc.values})
    eda_categorias['pct'] = 100 * eda_categorias.n_incidentes / len(df)
    salvar_parquet(eda_categorias, 'eda_categorias')

eda_resumo = pd.DataFrame([
    {'grupo': 'CAUSA (entrada do modelo)', 'n_colunas': len(cause_cols),
     'colunas': ', '.join(cause_cols)},
    {'grupo': 'EFEITO (so interpretacao)', 'n_colunas': len(effect_cols),
     'colunas': ', '.join(effect_cols)},
    {'grupo': 'EXCLUIR (leakage/NULL)', 'n_colunas': len(exclude_cols),
     'colunas': ', '.join(exclude_cols)},
])
salvar_parquet(eda_resumo, 'eda_resumo_features')


[STATS] ESTRUTURA DE FEATURES:
   [OK] CAUSA (incluir): 14 colunas
   [WARN] EFEITO (interpretar): 7 colunas
   [ERROR] EXCLUIR: 6 colunas
   Total dataset: 26
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_cardinalidade.parquet  (8 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_estatisticas_causa.parquet  (5 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_correlacao_causa.parquet  (25 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_nulos.parquet  (3 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_categorias.parquet  (141 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/eda_resumo_features.parquet  (3 linhas)


PosixPath('/home/fiap/mvp-locaweb/data/ml/kmeans/eda_resumo_features.parquet')

In [5]:
# ===== [4] FEATURE ENGINEERING: FREQUENCY ENCODING (OTIMIZADO PARA K-MEANS) =====
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("\n🔧 FEATURE ENGINEERING: FREQUENCY ENCODING (SEM ONE-HOT ENCODING)")

# 1. Tratamento de Outliers (Lógica original mantida)
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [
    c for c in numeric_cols if c not in ["incident_id", "cluster", "data_abertura"]
]

outlier_counts = pd.DataFrame(index=df.index)
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 * IQR
  outlier_counts[col] = ~((df[col] >= lower) & (df[col] <= upper))

outlier_total_per_row = outlier_counts.sum(axis=1)
max_outliers_allowed = len(numeric_cols) * 0.5
outlier_mask = outlier_total_per_row <= max_outliers_allowed

df_clean = df[outlier_mask].copy()

# 2. Seleção de Features de Causa
cause_cols = [
    "prioridade_num",
    "grupo_designado",
    "categoria",
    "subcategoria",
    "produto",
    "hora_abertura",
    "turno_abertura",
    "dia_semana_num",
    "fora_horario_comercial",
    "abriu_fim_de_semana",
    "mes_abertura",
    "trimestre",
    "possui_pai",
    "triagem_incompleta",
]
cause_cols_valid = [c for c in cause_cols if c in df_clean.columns]
df_features = df_clean[cause_cols_valid].copy()

# 3. Encoding Cíclico para Features Temporais
temporal_cyclic = {
    "hora_abertura": 24,
    "dia_semana_num": 7,
    "mes_abertura": 12,
}
for col, period in temporal_cyclic.items():
  if col in df_features.columns:
    df_features[f"{col}_sin"] = np.sin(2 * np.pi * df_features[col] / period)
    df_features[f"{col}_cos"] = np.cos(2 * np.pi * df_features[col] / period)
    df_features = df_features.drop(columns=[col])

# 4. Frequency Encoding para TODAS as categóricas (Evita a explosão de colunas)
all_categorical_cols = [
    "grupo_designado",
    "subcategoria",
    "produto",
    "categoria",
    "turno_abertura",
]
for col in all_categorical_cols:
  if col in df_features.columns:
    freq_map = df_features[col].value_counts(normalize=True).to_dict()
    df_features[col] = df_features[col].map(freq_map).fillna(0)

# 5. Normalização com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features.fillna(0))

print(
    f"✅ Features Finais (sem One-Hot Encoding): {X_scaled.shape[1]} colunas"
    " numéricas contínuas."
)


🔧 FEATURE ENGINEERING: FREQUENCY ENCODING (SEM ONE-HOT ENCODING)


✅ Features Finais (sem One-Hot Encoding): 17 colunas numéricas contínuas.


In [6]:
# ===== [4b] IMPACTO DA SELECAO DE FEATURES (em parquet) =====
FEATURES_ANTES = 680   # explosao de one-hot da versao original do notebook

impacto = pd.DataFrame([
    {'etapa': 'antes (OHE explosion)', 'n_features': FEATURES_ANTES},
    {'etapa': 'depois (CAUSA only)',   'n_features': int(X_scaled.shape[1])},
    {'etapa': 'removidas',             'n_features': FEATURES_ANTES - int(X_scaled.shape[1])},
])
impacto['pct_do_total_antes'] = 100 * impacto.n_features / FEATURES_ANTES
salvar_parquet(impacto, 'feature_selection_impacto')

# Correlacao pos-selecao, formato longo (substitui o heatmap)
correlacao_pos = (pd.DataFrame(X_scaled).corr()
                  .stack().rename('correlacao').reset_index()
                  .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'}))
correlacao_pos[['feature_a', 'feature_b']] = correlacao_pos[['feature_a', 'feature_b']].astype(str)
salvar_parquet(correlacao_pos, 'correlacao_pos_selecao')

print(f'Features: {FEATURES_ANTES} -> {X_scaled.shape[1]}')


   -> /home/fiap/mvp-locaweb/data/ml/kmeans/feature_selection_impacto.parquet  (3 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/correlacao_pos_selecao.parquet  (289 linhas)
Features: 680 -> 17


In [7]:
# ===== [4c] PCA: REDUÇÃO DIMENSIONAL (95% variância) =====
from sklearn.decomposition import PCA

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
n_components = pca.n_components_
explained_var = sum(pca.explained_variance_ratio_)

print(f'\n📉 PCA: REDUÇÃO DIMENSIONAL')
print(f'   Features antes: {X_scaled.shape[1]}')
print(f'   Componentes após: {n_components}')
print(f'   Variância explicada: {explained_var*100:.2f}%')
print(f'   Shape pós-PCA: {X_pca.shape}')


📉 PCA: REDUÇÃO DIMENSIONAL
   Features antes: 17
   Componentes após: 3
   Variância explicada: 39.95%
   Shape pós-PCA: (41441, 3)


In [8]:
# ===== [5] K-MEANS TRAINING (no espaço PCA) =====
k = 4

# Verificar se PCA foi executado (célula 4c)
if 'X_pca' not in locals():
    print('⚠️ Aviso: X_pca não definido. Usando X_scaled diretamente.')
    X_train = X_scaled
    n_components = X_scaled.shape[1]
else:
    X_train = X_pca
    if 'n_components' not in locals():
        n_components = X_pca.shape[1]

model = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
print(f'\n🤖 Treinando K-Means com k={k} (espaço com {n_components} dimensões)...')

try:
    labels = model.fit_predict(X_train)

    sil_score = silhouette_score(X_train, labels)
    db_score = davies_bouldin_score(X_train, labels)

    print(f'\n📊 Métricas de Clustering:')
    print(f'   Silhouette Score: {sil_score:.4f}')
    print(f'   Davies-Bouldin Index: {db_score:.4f}')
    print(f'   Dataset: {len(labels):,} amostras')
    print(f'   ✅ Modelo KMeans treinado com sucesso')
except Exception as e:
    print(f'❌ Erro ao treinar KMeans: {str(e)}')
    print(f'   Verifique se as células anteriores foram executadas corretamente')
    raise


🤖 Treinando K-Means com k=4 (espaço com 3 dimensões)...



📊 Métricas de Clustering:
   Silhouette Score: 0.3413
   Davies-Bouldin Index: 0.9876
   Dataset: 41,441 amostras
   ✅ Modelo KMeans treinado com sucesso


In [9]:
# ===== [6a] ATRIBUIÇÃO DE CLUSTERS (limpos + outliers) =====
# Adicionar clusters ao dataset limpo
df_clean['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df_clean['cluster_label'] = df_clean['cluster_pred'].map(cluster_names)

print(f'\n✅ Clusters atribuídos ao dataset limpo ({len(df_clean):,} registros)')

# ===== PREDIZER CLUSTERS PARA OS OUTLIERS REMOVIDOS =====
df_outliers = df[~outlier_mask].copy() if 'outlier_mask' in locals() else pd.DataFrame()

if len(df_outliers) > 0:
    print(f'\n🔄 Atribuindo clusters aos {len(df_outliers):,} incidentes removidos como outliers...')

    try:
        # Define as colunas categoricas que foram usadas para OneHotEncoding
        low_cardinality_cols = ['categoria', 'turno_abertura']
        high_cardinality_cols = ['grupo_designado', 'subcategoria', 'produto']
        temporal_cyclic_cols = ['hora_abertura', 'dia_semana_num', 'mes_abertura']

        # Preparar features dos outliers (mesmo processo que df_clean)
        X_outliers = df_outliers[cause_cols_valid].copy() if 'cause_cols_valid' in locals() else df_outliers.copy()

        # 1. Cyclical encoding para features temporais
        for col, period in zip(temporal_cyclic_cols, [24, 7, 12]):
            if col in X_outliers.columns:
                X_outliers[f'{col}_sin'] = np.sin(2 * np.pi * X_outliers[col] / period)
                X_outliers[f'{col}_cos'] = np.cos(2 * np.pi * X_outliers[col] / period)

        # 2. Frequency encoding para alta cardinalidade (calculado a partir de df_clean)
        for col in high_cardinality_cols:
            if col in X_outliers.columns:
                # Usar frequências do dataset limpo
                freq_map = df_clean[col].value_counts(normalize=True).to_dict()
                X_outliers[col] = X_outliers[col].map(freq_map).fillna(0)

        # 3. One-Hot Encoding com mesmo encoder
        if 'encoder' in locals() and low_cardinality_cols:
            for col in low_cardinality_cols:
                if col in X_outliers.columns:
                    X_outliers[col] = X_outliers[col].fillna('unknown').astype(str)

            X_outliers_cat = encoder.transform(X_outliers[low_cardinality_cols])
            X_outliers_cat_df = pd.DataFrame(X_outliers_cat, columns=encoder.get_feature_names_out(low_cardinality_cols))

            # Selecionar apenas as colunas numericas que foram usadas
            numeric_cols_use = [c for c in X_outliers.columns if c not in low_cardinality_cols and
                               X_outliers[c].dtype in ['int64', 'float64']]
            X_outliers_num = X_outliers[numeric_cols_use].reset_index(drop=True)
            X_outliers_proc = pd.concat([X_outliers_num, X_outliers_cat_df.reset_index(drop=True)], axis=1)
        else:
            X_outliers_proc = X_outliers

        # 4. Escalar com mesmo scaler
        if 'scaler' in locals():
            X_outliers_scaled = scaler.transform(X_outliers_proc.fillna(0))
        else:
            X_outliers_scaled = X_outliers_proc.fillna(0)

        # 5. Aplicar PCA com mesmo transformer
        if 'pca' in locals() and 'X_pca' in locals():
            X_outliers_pca = pca.transform(X_outliers_scaled)
            labels_outliers = model.predict(X_outliers_pca)
        else:
            labels_outliers = model.predict(X_outliers_scaled)

        df_outliers['cluster_pred'] = labels_outliers
        df_outliers['cluster_label'] = df_outliers['cluster_pred'].map(cluster_names)

        print(f'   ✅ Clusters preditos para outliers')
    except Exception as e:
        print(f'   ⚠️ Erro ao predizer clusters para outliers: {str(e)}')
        print(f'   Continuando apenas com dataset limpo')
        df_outliers = pd.DataFrame()
else:
    print('   ℹ️ Nenhum outlier removido, pulando etapa')

# ===== COMBINAR TODOS OS INCIDENTES =====
if len(df_outliers) > 0:
    df_final = pd.concat([df_clean, df_outliers], ignore_index=False).sort_index()
else:
    df_final = df_clean.copy()

print(f'\n📋 Distribuição Final de Clusters:')
cluster_dist = df_final['cluster_label'].value_counts().sort_index()
for label in ['A', 'B', 'C', 'D']:
    count = cluster_dist.get(label, 0)
    pct = count / len(df_final) * 100 if count > 0 else 0
    print(f'   Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')

print(f'\n✅ Dataset Final:')
print(f'   Total: {len(df_final):,} incidentes (100.00%)')
print(f'   Todos os incidentes receberam cluster label')


✅ Clusters atribuídos ao dataset limpo (41,441 registros)
   ℹ️ Nenhum outlier removido, pulando etapa

📋 Distribuição Final de Clusters:
   Cluster A:   13,289 incidentes (32.07%)
   Cluster B:   13,625 incidentes (32.88%)
   Cluster C:   10,175 incidentes (24.55%)
   Cluster D:    4,352 incidentes (10.50%)

✅ Dataset Final:
   Total: 41,441 incidentes (100.00%)
   Todos os incidentes receberam cluster label


In [10]:
# ===== [6b] CLUSTER PROFILING: INTERPRETAR COM FEATURES EFEITO =====
print('[INFO] CLUSTER PROFILING: Interpretacao dos Clusters')

# Features de EFEITO para interpretar clusters
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# Usar colunas de EFEITO que já existem em df_final (herdadas de df_clean)
effect_cols_available = [c for c in effect_cols if c in df_final.columns]
df_with_effect = df_final.copy()

# ===== Perfil dos clusters em parquet (substitui os 4 paineis da figura) =====
tamanhos = df_final['cluster_label'].value_counts().sort_index()
distribuicao = pd.DataFrame({'cluster_label': tamanhos.index,
                             'n_incidentes': tamanhos.values})
distribuicao['pct'] = 100 * distribuicao.n_incidentes / len(df_final)

if 'foi_resolvido' in df_with_effect.columns:
    taxa = df_with_effect.groupby('cluster_label')['foi_resolvido'].mean() * 100
    distribuicao['taxa_resolucao_pct'] = distribuicao.cluster_label.map(taxa)
if 'horas_ate_resolucao' in df_with_effect.columns:
    media_h = df_with_effect.groupby('cluster_label')['horas_ate_resolucao'].mean()
    distribuicao['tempo_medio_horas'] = distribuicao.cluster_label.map(media_h)
if 'excedeu_tempo_esperado' in df_with_effect.columns:
    sla = df_with_effect.groupby('cluster_label')['excedeu_tempo_esperado'].mean() * 100
    distribuicao['taxa_sla_violado_pct'] = distribuicao.cluster_label.map(sla)
salvar_parquet(distribuicao, 'cluster_distribuicao')

# Quartis por cluster: e a informacao que o boxplot mostrava
if 'horas_ate_resolucao' in df_with_effect.columns:
    quartis = (df_with_effect.groupby('cluster_label')['horas_ate_resolucao']
               .describe()[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
               .reset_index()
               .rename(columns={'25%': 'q1', '50%': 'mediana', '75%': 'q3'}))
    salvar_parquet(quartis, 'cluster_quartis_resolucao')

# Perfil normalizado por metrica de EFEITO: valor medio e o mesmo valor em escala 0-1
numeric_effect = [c for c in effect_cols_available
                  if c in df_with_effect.columns and df_with_effect[c].dtype in ['int64', 'float64']]
if numeric_effect:
    perfil = df_with_effect.groupby('cluster_label')[numeric_effect].mean()
    perfil_norm = (perfil - perfil.min()) / (perfil.max() - perfil.min() + 1e-9)
    perfil_longo = (perfil.stack().rename('valor_medio').reset_index()
                    .rename(columns={'level_1': 'metrica'}))
    perfil_longo['valor_normalizado'] = (perfil_norm.stack()
                                         .reset_index(drop=True).values)
    salvar_parquet(perfil_longo, 'cluster_perfil_efeito')

print('[OK] Perfil dos clusters gravado em parquet')

# Mostrar resumo interpretativo
print('[INFO] INTERPRETACAO DOS CLUSTERS:')
for label in sorted(df_final['cluster_label'].unique()):
    mask = df_final['cluster_label'] == label
    count = mask.sum()
    pct = count/len(df_final)*100
    print(f'  Cluster {label}: {count:,} incidentes ({pct:.1f}%)')

    if 'horas_ate_resolucao' in df_with_effect.columns:
        avg_duration = df_with_effect[mask]['horas_ate_resolucao'].mean()
        print(f'           Tempo medio: {avg_duration:.1f} horas')

    if 'excedeu_tempo_esperado' in df_with_effect.columns:
        sla_violation = df_with_effect[mask]['excedeu_tempo_esperado'].mean() * 100
        print(f'           Taxa SLA violado: {sla_violation:.1f}%')

[INFO] CLUSTER PROFILING: Interpretacao dos Clusters
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/cluster_distribuicao.parquet  (4 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/cluster_quartis_resolucao.parquet  (4 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/cluster_perfil_efeito.parquet  (16 linhas)
[OK] Perfil dos clusters gravado em parquet
[INFO] INTERPRETACAO DOS CLUSTERS:
  Cluster A: 13,289 incidentes (32.1%)
           Tempo medio: 30.8 horas
           Taxa SLA violado: 94.8%
  Cluster B: 13,625 incidentes (32.9%)
           Tempo medio: 198.3 horas
           Taxa SLA violado: 95.2%
  Cluster C: 10,175 incidentes (24.6%)
           Tempo medio: 20.8 horas
           Taxa SLA violado: 94.3%
  Cluster D: 4,352 incidentes (10.5%)
           Tempo medio: 93.8 horas
           Taxa SLA violado: 98.0%


In [11]:
# ===== [7] EXPORTACAO DOS ARTEFATOS (parquet em data/ml) =====
# Garante que a coluna de clusters seja atribuida
df_final['cluster_id'] = df_final['cluster_label']

salvar_parquet(df_final, 'base_incidentes_clusterizada')

# Perfil por cluster: o que o BI consome para descrever cada grupo
perfil_cols = [c for c in effect_cols_available if c in df_final.columns]
perfil_clusters = (df_final.groupby('cluster_label')
                   .agg(n_incidentes=('cluster_label', 'size'),
                        **{c: (c, 'mean') for c in perfil_cols})
                   .reset_index())
salvar_parquet(perfil_clusters, 'perfil_clusters')

# Metricas do modelo, no mesmo formato longo dos demais notebooks
metricas_kmeans = pd.DataFrame([
    {'modelo_nome': 'clustering_kmeans', 'modelo_tipo': 'clustering',
     'metrica': 'silhouette', 'valor': float(sil_score)},
    {'modelo_nome': 'clustering_kmeans', 'modelo_tipo': 'clustering',
     'metrica': 'davies_bouldin', 'valor': float(db_score)},
    {'modelo_nome': 'clustering_kmeans', 'modelo_tipo': 'clustering',
     'metrica': 'k', 'valor': float(k)},
    {'modelo_nome': 'clustering_kmeans', 'modelo_tipo': 'clustering',
     'metrica': 'n_amostras', 'valor': float(len(df_final))},
])
metricas_kmeans['data_metrica'] = pd.Timestamp.now().normalize()
salvar_parquet(metricas_kmeans, 'fct_model_metrics')

print(f'\nArtefatos gravados em: {OUT_DIR}')


   -> /home/fiap/mvp-locaweb/data/ml/kmeans/base_incidentes_clusterizada.parquet  (41,441 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/perfil_clusters.parquet  (4 linhas)
   -> /home/fiap/mvp-locaweb/data/ml/kmeans/fct_model_metrics.parquet  (4 linhas)

Artefatos gravados em: /home/fiap/mvp-locaweb/data/ml/kmeans


In [ ]:
# ===== [8] PERSISTENCIA EM ml.fct_perfil_cluster (banco fiap) =====
from sqlalchemy import text
import hashlib

data_execucao = pd.Timestamp.now()

perfil_db = distribuicao.rename(columns={
    'cluster_label': 'cluster_id',
    'pct': 'pct_volume',
    'tempo_medio_horas': 'duracao_media_horas',
    'taxa_sla_violado_pct': 'taxa_excedeu_tempo_esperado_pct',
}).copy()
perfil_db['modelo_versao'] = MODELO_VERSAO_CLUSTER
perfil_db['data_execucao'] = data_execucao
perfil_db['perfil_cluster_sk'] = [
    hashlib.md5(f'{c}|{MODELO_VERSAO_CLUSTER}|{data_execucao}'.encode()).hexdigest()
    for c in perfil_db['cluster_id']
]

cols = ['perfil_cluster_sk', 'cluster_id', 'modelo_versao', 'data_execucao',
        'n_incidentes', 'pct_volume', 'duracao_media_horas', 'taxa_resolucao_pct',
        'taxa_excedeu_tempo_esperado_pct']
for c in cols:
    if c not in perfil_db.columns:
        perfil_db[c] = None
perfil_db = perfil_db[cols]

# GUARDA (Fase 5): antes de sobrescrever o perfil ja em producao e ja
# verificado pela auditoria, confirmar que o rerun reproduziu os mesmos
# numeros (pipeline deterministico: random_state=42 em toda etapa
# estocastica). Se divergir, abortar em vez de sobrescrever silenciosamente.
perfil_atual = pd.read_sql(
    "SELECT cluster_id, pct_volume, taxa_excedeu_tempo_esperado_pct FROM ml.fct_perfil_cluster ORDER BY cluster_id",
    engine,
)
perfil_novo_cmp = perfil_db[['cluster_id', 'pct_volume', 'taxa_excedeu_tempo_esperado_pct']].sort_values('cluster_id').reset_index(drop=True)
perfil_atual_cmp = perfil_atual.sort_values('cluster_id').reset_index(drop=True)

if len(perfil_atual_cmp) > 0:
    diff = (perfil_novo_cmp.set_index('cluster_id') - perfil_atual_cmp.set_index('cluster_id')).abs()
    max_diff = diff.max().max()
    if max_diff > 0.5:
        print('ATENCAO: rerun divergiu do perfil em producao em mais de 0.5pp:')
        print(diff)
        raise RuntimeError(
            'Rerun do K-Means divergiu do ml.fct_perfil_cluster existente -- '
            'abortando persistencia para nao sobrescrever silenciosamente. '
            'Investigar antes de prosseguir.'
        )
    else:
        print(f'Rerun consistente com ml.fct_perfil_cluster existente (divergencia max: {max_diff:.4f}pp)')

with engine.begin() as conn:
    conn.execute(text('TRUNCATE ml.fct_perfil_cluster'))
perfil_db.to_sql('fct_perfil_cluster', engine, schema='ml', if_exists='append', index=False)
print(f'ml.fct_perfil_cluster: {len(perfil_db)} linhas (clusters: {list(perfil_db.cluster_id)})')


In [ ]:
# ===== [9] PERSISTENCIA DO cluster_id POR INCIDENTE (Fase 5) =====
# ml.ml_cluster_dataset.cluster_id sempre existiu como coluna mas nunca foi
# preenchida (achado da Fase 4.2: 100% NULL -- o assignment por incidente
# nunca tinha sido persistido, so o perfil agregado em fct_perfil_cluster
# existia). Preenche agora que o rerun foi validado (celula anterior) contra
# o perfil ja em producao. Sem isso, a composicao por prioridade/categoria
# da Tela 04 fica sem fonte.
from psycopg2.extras import execute_batch

assert 'incident_id' in df_final.columns, 'df_final perdeu incident_id -- nao persistir'
mapa = df_final[['incident_id', 'cluster_id']].dropna(subset=['incident_id', 'cluster_id'])
print(f'Persistindo cluster_id para {len(mapa):,} incidentes...')

raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor()
    execute_batch(
        cur,
        'UPDATE ml.ml_cluster_dataset SET cluster_id = %s WHERE incident_id = %s',
        [(row.cluster_id, row.incident_id) for row in mapa.itertuples()],
        page_size=2000,
    )
    raw_conn.commit()
finally:
    raw_conn.close()

restantes = pd.read_sql("SELECT count(*) as n FROM ml.ml_cluster_dataset WHERE cluster_id IS NULL", engine)
print(f'Persistido. Incidentes ainda sem cluster_id: {restantes.iloc[0].n}')
